# EmotWen — Step 4: GRPO Length Control (Conditional)

**Run this notebook only if `03_eval.ipynb` reported `grpo_needed=True`**
(i.e., >15% of responses exceed 5 sentences).

Uses GRPO with two reward functions:
- `length_reward`: targets 2–5 sentences, with `'Let me explain:'` exception
- `advice_penalty_reward`: penalises advice-giving language

After training, automatically re-runs evaluation and logs a comparison to W&B.

**Prerequisites:** Run `02_sft_train.ipynb` and `03_eval.ipynb` first.

> ⚠ Training takes ~75–100 min on T4. `save_steps=20` ensures checkpoints survive session timeouts.

In [ ]:
%%capture
import os, importlib.util, sys
!pip install --upgrade -qqq uv
if importlib.util.find_spec('torch') is None or 'COLAB_' in ''.join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = 'numpy'; _pil = 'pillow'
    !uv pip install -qqq \
        'torch==2.8.0' 'triton>=3.3.0' {_numpy} {_pil} bitsandbytes 'xformers==0.0.32.post2' \
        'unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo' \
        'unsloth[base] @ git+https://github.com/unslothai/unsloth'
elif importlib.util.find_spec('unsloth') is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers 'trl==0.22.2' unsloth unsloth_zoo
!uv pip install 'transformers==5.2.0'
!uv pip install --no-build-isolation flash-linear-attention 'causal_conv1d==1.6.0'
!uv pip install wandb datasets nltk

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/emotwen-3.5-finetune'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/afonsomota/emotwen-3.5-finetune.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
except ImportError:
    REPO_DIR = os.path.dirname(os.getcwd())

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import nltk
nltk.download('punkt_tab', quiet=True)

In [ ]:
import wandb
wandb.login()

In [ ]:
# ── Config overrides ──────────────────────────────────────────────────────────
CONFIG = {
    'wandb_project': 'emotwen-journal-chat',
    'run_name': 'grpo_v1',

    # SFT adapter to start from
    'sft_adapter_path': os.path.join(REPO_DIR, 'outputs', 'sft_stage2'),

    # GRPO output dirs
    'output_dir': os.path.join(REPO_DIR, 'outputs', 'grpo_adapter'),
    'final_merged_dir': os.path.join(REPO_DIR, 'outputs', 'final_merged'),

    # Set skip_if_not_needed=True to auto-abort if eval said grpo_needed=False
    'skip_if_not_needed': False,

    # Training steps (reduce for quick debug)
    # 'max_steps': 20,

    # Number of prompts to train on
    'n_grpo_prompts': 400,
}

In [ ]:
# ── Run GRPO ──────────────────────────────────────────────────────────────────
from src.train_grpo import run

results = run(CONFIG)

if results.get('grpo_skipped'):
    print('GRPO was skipped (model already within length bounds).')
else:
    print('\n── GRPO Results ─────────────────────────────────────────')
    print(f'  Pre-GRPO  pct_in_range: {results["pre_grpo_pct_in_range"]}')
    print(f'  Post-GRPO pct_in_range: {results["post_grpo_pct_in_range"]:.1%}')
    print(f'  Post-GRPO advice_rate:  {results["post_grpo_advice_rate"]:.1%}')
    print(f'  Still needs GRPO:       {results["post_grpo_grpo_still_needed"]}')
    print(f'  Final merged model:     {results["final_merged_dir"]}')

In [ ]:
# ── GPU memory summary ────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    used = torch.cuda.max_memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'Peak GPU memory: {used:.2f} GB / {total:.2f} GB ({100*used/total:.1f}%)')